# Pré-processamento dos dados

---

## 1. Carregando e separando os dados

In [12]:
RANDOM_SEED = 1

In [13]:
import pandas as pd

leads = pd.read_csv('data/churn_aplicacao_leads_final.csv')
pedidos = pd.read_csv('data/churn_comportamental_pedidos_final.csv')

total = leads.merge(right=pedidos, on='id_cliente', how='left')

Vamos treinar dois modelos: um **comportamental** e um de **análise de leads**.

No modelo comportamental, o conjunto de treinamento será referente aos clientes que tem pedidos correspondentes na tabela de pedidos, enquanto que o conjunto de treinamento do modelo de análise de leads conterá apenas dados de clientes que nunca fizeram um pedido antes. Nesse último caso, nenhum atributo da tabela de pedidos será relevante, então podemos também removê-los dos atributos disponíveis.

Com base nisso, separamos os conjuntos usando essas regras:

In [14]:
leads_raw: pd.DataFrame = total[total.id_pedido.isna()]           # Clientes que nunca pediram
comportamental_raw: pd.DataFrame = total[total.id_pedido.notna()] # Clientes que tem pedidos

leads_raw = leads_raw.drop(['id_pedido', 'id_cliente', 'valor', 'produtos', 'data_entrega'], axis='columns') # Removendo atributos referentes a pedidos

Por fim, vamos definir uma porcentagem para a divisão entre treino + validação e teste:

In [15]:
TRAIN_VAL_RATIO = 0.8
TEST_RATIO      = 0.2

### 1.1. Separando o conjunto de dados `leads_raw`

Sem levar os pedidos em consideração, temos um conjunto de dados que não apresenta uma relação de temporalidade relevante, então a sua divisão segue de modo padrão: faremos uma divisão estratificada no atributo-alvo, o `churned`:

In [16]:
from sklearn.model_selection import train_test_split

leads_raw_X, leads_raw_y = leads_raw, leads_raw.pop('churned')

leads_raw_train_val_X, leads_raw_test_X, leads_raw_train_val_y, leads_raw_test_y = train_test_split(
    leads_raw_X,
    leads_raw_y,
    train_size=TRAIN_VAL_RATIO,
    test_size=TEST_RATIO,
    random_state=RANDOM_SEED,
    stratify=leads_raw_y
)

LEADS = pd.DataFrame({ # pondo tudo num DataFrame só pela facilidade de acesso
    'train_val': {
        'X': leads_raw_train_val_X,
        'y': leads_raw_train_val_y,
    },
    'test': {
        'X': leads_raw_test_X,
        'y': leads_raw_test_y,
    },
})

print(f'Tamanho do conjunto de treino + validação: {LEADS.train_val.X.shape[0]} linhas x {LEADS.train_val.X.shape[1]} colunas')
print(f'Tamanho do conjunto de teste:              {LEADS.test.X.shape[0]} linhas x {LEADS.test.X.shape[1]} colunas')

Tamanho do conjunto de treino + validação: 800 linhas x 17 colunas
Tamanho do conjunto de teste:              200 linhas x 17 colunas


### 1.2. Separando o conjunto de dados `comportamental_raw`

Como o conjunto de dados comportamental leva em consideração os pedidos dos clientes, temos uma série temporal, como discutido no processo de EDA.

Para separar os dados agora, é necessário evitar que o futuro seja usado como treinamento para testes no passado, então, devemos ordenar os dados com base na data. A partir disso, basta escolher uma data de corte e separar os dois conjuntos adequadamente.

Queremos manter a mesma proporção de amostras de teste que o conjunto de análise de leads, então usaremos `TEST_RATIO` do intervalo total (90 dias) para definir a janela de corte:

In [17]:
TEST_TIMEFRAME = pd.to_timedelta(90 * TEST_RATIO, 'days')

Agora, basta separar ambos conjuntos:

In [18]:
comportamental_raw.data_entrega = comportamental_raw.data_entrega.apply(pd.to_datetime)

ordered = comportamental_raw.sort_values(by='data_entrega')

date_threshold = pd.to_datetime(ordered.data_entrega.max()) - TEST_TIMEFRAME

comportamental_raw_train_val = ordered[ordered.data_entrega < date_threshold]
comportamental_raw_test = ordered[ordered.data_entrega >= date_threshold]

COMPORTAMENTAL = pd.DataFrame({ # pondo tudo num DataFrame só pela facilidade de acesso
    'train_val': {
        'X': comportamental_raw_train_val,
        'y': comportamental_raw_train_val.pop('churned'),
    },
    'test': {
        'X': comportamental_raw_test,
        'y': comportamental_raw_test.pop('churned'),
    },
})

print(f'Tamanho do conjunto de treino + validação: {COMPORTAMENTAL.train_val.X.shape[0]} linhas x {COMPORTAMENTAL.train_val.X.shape[1]} colunas')
print(f'Tamanho do conjunto de teste:              {COMPORTAMENTAL.test.X.shape[0]}  linhas x {COMPORTAMENTAL.test.X.shape[1]} colunas')

Tamanho do conjunto de treino + validação: 3554 linhas x 22 colunas
Tamanho do conjunto de teste:              887  linhas x 22 colunas


Verificando as proporções finais e os intervalos de tempo a serem utilizados:

In [19]:
print('Porcentagem da divisão dos dados comportamentais:')

print(f'{100 * COMPORTAMENTAL.train_val.X.shape[0] / comportamental_raw.shape[0]:.2f}% para treino e validação')
print(f'       |-> Intervalo de tempo: {90 - TEST_TIMEFRAME.days} dias')

print(f'{100 * COMPORTAMENTAL.test.X.shape[0] / comportamental_raw.shape[0]:.2f}% para teste')
print(f'       |-> Intervalo de tempo: {TEST_TIMEFRAME.days} dias')

Porcentagem da divisão dos dados comportamentais:
80.03% para treino e validação
       |-> Intervalo de tempo: 72 dias
19.97% para teste
       |-> Intervalo de tempo: 18 dias


---

# 2. Features e pré-processamento

O que falta pra chegar nos modelos:

1. Transformar os dados **cadastrais** (transformações fixas, linha a linha)
2. Transformar os **pedidos** do modelo comportamental em features por cliente
3. Montar o **pré-processamento** que aprende só no treino
4. Definir a **validação cruzada** de cada modelo
5. Rodar um teste de fumaça

In [20]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

### 2.1. Dados cadastrais

Transformações **fixas** (uma regra aplicada linha a linha, não aprendem nada dos dados), então podem ser aplicadas igual nos dois modelos e em treino/teste:

| coluna original | vira | por quê |
|---|---|---|
| `capital_social` (faixa) | `capital_social_ord` | faixas em ordem, do menor pro maior capital, viram o código 0, 1, 2, ...  |
| `idade_cnpj` (faixa, em dias) | `idade_cnpj_ord` | faixas em ordem, da menor pra maior idade, viram o código 0, 1, 2, ... |
| `serasa_contagem_negativacoes` / `_protestos` | `negativacoes_log` / `protestos_log` | têm valores muito altos - são outliers **estatísticos** (podem ser reais): mantemos e usamos `log1p` |
| `serasa_credores` (lista) | `qtd_credores` | quantos itens tem na lista (0 se nulo) |
| `serasa_credores` (lista) | `credor_<tipo>` | cada tipo da lista (separados por vírgula) vira uma coluna: 1 se o cliente tem o tipo, 0 caso contrário. Tipos com menos de 30 clientes no treino são juntados em `credor_Outros` (ver 2.4) |
| `natureza_juridica` (4 categorias, 2 quase vazias) | `eh_sociedade` | junta as 2 individuais e as 2 sociedades |
| `cnae_codigo` | `cnae_divisao` | só os 2 primeiros dígitos, tratado como **categoria** |
| `ifood_*` / `google_maps_*` | `tem_ifood` / `tem_google_maps` | quase tudo faltante: viram só "tem ou não" |

Ficam de fora: `uf` (só tem CE), `municipio` (a EDA não achou relação com churn) e os detalhes de iFood/Google Maps.

In [21]:
def limite_inferior(faixa):
    # '(3000, 5000]' -> 3000.0
    return float(faixa.strip('()[]').split(',')[0])


def ordem_das_faixas(serie):
    # lista das faixas que existem na coluna, da menor pra maior
    return sorted(serie.dropna().unique(), key=limite_inferior)


ORDEM_CAPITAL = ordem_das_faixas(leads['capital_social'])
print(f'{len(ORDEM_CAPITAL)} faixas de capital social: {ORDEM_CAPITAL}')

ORDEM_IDADE = ordem_das_faixas(leads['idade_cnpj'])
print(f'{len(ORDEM_IDADE)} faixas de idade do CNPJ: {ORDEM_IDADE}')


def tipos_de_credor(lista):
    # 'Alimentos e Bebidas, Saúde' -> {'Alimentos e Bebidas', 'Saúde'}
    if pd.isna(lista):
        return set()
    return set(lista.split(', '))


TIPOS_CREDOR = sorted(set().union(*leads['serasa_credores'].map(tipos_de_credor)))
print(f'{len(TIPOS_CREDOR)} tipos de credor: {TIPOS_CREDOR}')


def prepara_cadastro(df):
    # recebe as colunas cadastrais originais e devolve as features cadastrais
    tipos = df['serasa_credores'].map(tipos_de_credor)

    return pd.DataFrame({
        # categoricas - agrupamento e encoding ficam pro pipeline (aprendem só no treino)
        'fonte_cliente': df['fonte_cliente'],
        'segmento_cliente': df['segmento_cliente'],
        'cnae_divisao': df['cnae_codigo'].str[:2],

        # ordinal -  posição da faixa na lista ordenada (0 = menor faixa)
        'capital_social_ord': df['capital_social'].map({faixa: i for i, faixa in enumerate(ORDEM_CAPITAL)}),
        'idade_cnpj_ord': df['idade_cnpj'].map({faixa: i for i, faixa in enumerate(ORDEM_IDADE)}),

        # numéricas
        'negativacoes_log': np.log1p(df['serasa_contagem_negativacoes']),
        'protestos_log': np.log1p(df['serasa_contagem_protestos']),
        'qtd_credores': df['serasa_credores'].str.split(', ').str.len().fillna(0),

        # binárias (0/1)
        'eh_sociedade': df['natureza_juridica'].str[:5].isin(['206-2', '224-0']).astype(int),
        'socio_negativado': df['serasa_socio_tem_negativacao'].astype(int),
        'tem_ifood': df['ifood_faixa_preco'].notna().astype(int),
        'tem_google_maps': df['google_maps_tem_website'].notna().astype(int),

        # tipos de credor - 1 se o cliente tem o tipo, 0 caso contrário
        **{f'credor_{tipo}': tipos.map(lambda tipos_cliente, tipo=tipo: int(tipo in tipos_cliente)) for tipo in TIPOS_CREDOR},
    }, index=df.index)

19 faixas de capital social: ['(0, 1]', '(1, 200]', '(200, 1000]', '(1000, 1500]', '(1500, 2000]', '(2000, 3000]', '(3000, 5000]', '(5000, 8000]', '(8000, 10000]', '(10000, 15000]', '(15000, 20000]', '(20000, 30000]', '(30000, 50000]', '(50000, 70000]', '(70000, 100000]', '(100000, 200000]', '(200000, 1000000]', '(1000000, 10000000]', '(10000000, 100000000000]']
20 faixas de idade do CNPJ: ['(0, 25]', '(25, 50]', '(50, 100]', '(100, 150]', '(150, 250]', '(250, 350]', '(350, 450]', '(450, 600]', '(600, 750]', '(750, 900]', '(900, 1150]', '(1150, 1300]', '(1300, 1600]', '(1600, 2000]', '(2000, 2400]', '(2400, 3000]', '(3000, 4000]', '(4000, 5000]', '(5000, 8000]', '(8000, 25000]']
21 tipos de credor: ['Alimentos e Bebidas', 'Bancos e Financeiro', 'Comunicação', 'Comércio', 'Contabilidade', 'Distribuição', 'Diversos', 'Energia', 'Financeiro', 'Imobiliário', 'Indústria', 'Logística', 'Saúde', 'Seguros e Financeiro', 'Serviços', 'Serviços Ambientais', 'Tecnologia', 'Tecnologia e Pagamentos'

### 2.2. Modelo de leads (clientes sem pedido)

Só o cadastro. Validação cruzada padrão, estratificada por `churned`.

In [22]:
X_leads_train_val = prepara_cadastro(LEADS.train_val.X).reset_index(drop=True)
y_leads_train_val = LEADS.train_val.y.reset_index(drop=True)

X_leads_test = prepara_cadastro(LEADS.test.X).reset_index(drop=True)
y_leads_test = LEADS.test.y.reset_index(drop=True)

folds_leads = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED).split(X_leads_train_val, y_leads_train_val))

print(f'leads - treino+validação: {X_leads_train_val.shape} | teste: {X_leads_test.shape}')
X_leads_train_val.head()

leads - treino+validação: (800, 33) | teste: (200, 33)


,fonte_cliente,segmento_cliente,cnae_divisao,capital_social_ord,idade_cnpj_ord,negativacoes_log,protestos_log,qtd_credores,eh_sociedade,socio_negativado,...,credor_Logística,credor_Saúde,credor_Seguros e Financeiro,credor_Serviços,credor_Serviços Ambientais,credor_Tecnologia,credor_Tecnologia e Pagamentos,credor_Telecomunicações,credor_Varejo,credor_Viagens e Turismo
0,Fonte 1,Segmento 16,77,6,16,0.00000,0.000000,0.0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,Fonte 1,Segmento 18,39,12,5,2.70805,0.000000,0.0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,Fonte 5,Segmento 9,77,9,19,0.00000,0.000000,0.0,1,1,...,0,0,0,0,0,0,0,0,0,0
3,Fonte 2,Segmento 20,53,7,8,0.00000,0.000000,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Fonte 5,Segmento 20,53,8,19,0.00000,0.693147,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 2.3. Modelo comportamental (clientes com pedido)

#### Uma linha por cliente **por janela**

O alvo (`churned`) é **um por cliente**. Pra ter features de comportamento, resumimos os pedidos de cada cliente.

Pra não misturar passado com futuro, o resumo é feito **por janela de tempo**: as features de uma linha usam **só os pedidos daquela janela**. O mesmo cliente pode aparecer em janelas diferentes, com padrão de compra diferente em cada uma.

As janelas têm o mesmo tamanho de 18 dias e são contadas de trás pra frente a partir da data de corte:

```
janela 0     janela 1     janela 2     janela 3  |  janela 4 
 18 dias      18 dias      18 dias      18 dias  |  a partir de date_threshold
```

In [23]:
pedidos_train_val = COMPORTAMENTAL.train_val.X.copy()
pedidos_test = COMPORTAMENTAL.test.X.copy()

pedidos_train_val['churned'] = COMPORTAMENTAL.train_val.y
pedidos_test['churned'] = COMPORTAMENTAL.test.y

N_JANELAS_TREINO = 4
ULTIMA_JANELA = N_JANELAS_TREINO    # janela do teste

# bordas das janelas de treino+validação: [corte - 72 dias, corte - 54, corte - 36, corte - 18, corte]
bordas = [date_threshold - TEST_TIMEFRAME * k for k in range(N_JANELAS_TREINO, -1, -1)]

# número da janela de cada pedido (0 a 3); -1 = antes da primeira janela (fica de fora)
pedidos_train_val['janela'] = np.searchsorted(bordas, pedidos_train_val['data_entrega'], side='right') - 1
pedidos_test['janela'] = ULTIMA_JANELA

for j in range(N_JANELAS_TREINO):
    ini, fim = bordas[j], bordas[j + 1] - pd.Timedelta(days=1)
    print(f'janela {j}: {ini.date()} a {fim.date()} | {(pedidos_train_val["janela"] == j).sum():4d} pedidos')
print(f'janela {ULTIMA_JANELA}: a partir de {date_threshold.date()} | {len(pedidos_test):4d} pedidos (TESTE)')

print(f'pedidos fora das janelas: {(pedidos_train_val["janela"] < 0).sum()}')
pedidos_train_val = pedidos_train_val[pedidos_train_val['janela'] >= 0]

janela 0: 2024-09-02 a 2024-09-19 |  856 pedidos
janela 1: 2024-09-20 a 2024-10-07 |  719 pedidos
janela 2: 2024-10-08 a 2024-10-25 | 1105 pedidos
janela 3: 2024-10-26 a 2024-11-12 |  861 pedidos
janela 4: a partir de 2024-11-13 |  887 pedidos (TESTE)
pedidos fora das janelas: 13


### Features dos pedidos

Por (cliente, janela):

| feature | descrição |
|---|---|
| `n_pedidos` | quantidade de pedidos na janela |
| `intervalo_medio_dias` | frequência de compra: média de dias entre dias com pedido. `NaN` quando só há 1 dia com pedido |
| `valor_mais_frequente`, `produtos_mais_frequente` | faixa mais frequente do cliente na janela (empate: a faixa maior), como código ordinal 0, 1, 2, ... |

Os pedidos com `valor`/`produtos` nulos (3 no total) são ignorados na contagem da faixa mais frequente.

**Obs:** não foram incluídas a variação da frequência de compra nem a variação do valor ao longo do tempo. Nas janelas de 18 dias, a maioria das linhas tem um único pedido, então não tem como medir variação dentro de uma janela; seria necessário comparar cada janela com a anterior.

In [24]:
ORDEM_VALOR = ordem_das_faixas(pedidos['valor'])
ORDEM_PRODUTOS = ordem_das_faixas(pedidos['produtos'])
print('valor:', ORDEM_VALOR)
print('produtos:', ORDEM_PRODUTOS)


def mais_frequente(codigos):
    # código da faixa que mais aparece; empate -> a faixa maior
    contagem = codigos.dropna().value_counts()
    if contagem.empty:
        return np.nan
    return contagem[contagem == contagem.max()].index.max()


def intervalo_medio(datas):
    dias = datas.drop_duplicates().sort_values()     # só dias distintos, em ordem
    return dias.diff().dt.days.mean()                # NaN se só há 1 dia


def resume_pedidos(pedidos):
    # uma linha por (cliente, janela), com o cadastro, as features de pedido e o churned do cliente
    pedidos = pedidos.assign(
        valor_ord=pedidos['valor'].map({faixa: i for i, faixa in enumerate(ORDEM_VALOR)}),
        produtos_ord=pedidos['produtos'].map({faixa: i for i, faixa in enumerate(ORDEM_PRODUTOS)}),
    )

    grupos = pedidos.groupby(['id_cliente', 'janela'])
    linhas = grupos.agg(
        n_pedidos=('id_pedido', 'count'),
        churned=('churned', 'first'),     # é o mesmo em todos os pedidos do cliente
    )
    linhas['intervalo_medio_dias'] = grupos['data_entrega'].apply(intervalo_medio)
    linhas['valor_mais_frequente'] = grupos['valor_ord'].apply(mais_frequente)
    linhas['produtos_mais_frequente'] = grupos['produtos_ord'].apply(mais_frequente)
    linhas = linhas.reset_index()

    # o cadastro é igual em todos os pedidos do cliente - pega de uma linha só
    cadastro = prepara_cadastro(pedidos.drop_duplicates('id_cliente').set_index('id_cliente'))
    return linhas.join(cadastro, on='id_cliente')


comp_train_val = resume_pedidos(pedidos_train_val)
comp_test = resume_pedidos(pedidos_test)

print(f'comportamental - treino+validação: {comp_train_val.shape} | teste: {comp_test.shape}')
print(f'taxa de churn - treino+validação: {comp_train_val["churned"].mean():.3f} | teste: {comp_test["churned"].mean():.3f}')
comp_train_val.groupby('janela')['churned'].agg(['count', 'mean']).round(3)

valor: ['(0, 500]', '(500, 1000]', '(1000, 2000]', '(2000, 10000000]']
produtos: ['(0, 5]', '(5, 10]', '(10, 10000000]']
comportamental - treino+validação: (2022, 40) | teste: (479, 40)
taxa de churn - treino+validação: 0.319 | teste: 0.225


,count,mean
janela,,
0,472,0.350
1,456,0.298
2,598,0.339
3,496,0.286
